# TN0 — dựng lại kết quả MobiVital, rồi chứng minh pipeline của mình tương đương

Chạy một mạch trên Colab, khoảng 60 phút.

## Nguyên tắc

Phần MobiVital chạy **đúng lệnh trong README của họ**, từ trong thư mục repo của
họ, không qua lớp bọc nào:

```bash
python -m training.autoreg_training
python -m inference.mobivital_gen
python -m inference.evaluate -m YOUR_METHOD.txt
```

Chỉ xen thêm lệnh `mv` đổi tên bảng kết quả giữa hai lần chạy — vì
`mobivital_gen.py` dòng 121 luôn ghi ra đúng một tên
`{mode}_mobivital_pre_invert_{corr}.txt`, chạy hai lần cùng `corr` là đè mất lần
trước. Chính README của họ viết `-m YOUR_METHOD.txt`, tức là họ tính sẵn việc
người dùng tự đổi tên.

Cuối notebook in `git status` của repo MobiVital — phải trống.

## Bốn bậc

```
TN0a   cham bang MobiVital commit san      -> so voi bai bao 0.819
TN0b   checkpoint cua ho -> tu sinh bang   -> moc cho TN0.1
TN0c   train lai tu dau  -> tu sinh bang   -> tai lap cong thuc train
TN0.1  checkpoint cua ho -> code CUA MINH  -> phai trung TN0b 537/537
```

TN0.1 cần thiết vì code MobiVital chỉ chạy được LSTM — `mobivital_gen.py` dòng
152 ghi cứng `LSTMMultiStep(...)`. Muốn thử TCN phải viết bộ chọn kênh riêng,
rồi chứng minh nó cho ra đúng kết quả code gốc.

## Cần chạy trước

`notebooks/DATA_PREPARE.ipynb`, **trong cùng phiên Colab này**. TN0 cần cả CSV
thô (13 GB, code MobiVital đọc thẳng) lẫn `by_user/*.npz`. CSV thô không cất lên
Drive nên ô setup tự chạy lại nếu thiếu.

## 0. Setup

In [ ]:
import os
import subprocess


def run(command):
    """Chạy một lệnh shell, in ra những gì nó in."""
    result = subprocess.run(command, shell=True, capture_output=True, text=True)
    print((result.stdout + result.stderr).strip())


REPO = "/content/UWB_RADAR"

if os.path.exists(REPO + "/.git"):
    os.chdir(REPO)
    run("git pull -q origin main")
else:
    os.chdir("/content")
    run("rm -rf " + REPO)
    run("git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git " + REPO)
    run("git clone -q https://github.com/nesl/mobivital-public.git "
        + REPO + "/external/mobivital")
    run("pip install -q einops")

os.chdir(REPO)
run("git rev-parse --short HEAD")
run("git -C external/mobivital rev-parse --short HEAD")

### Dữ liệu

Thiếu thì ô dưới tải lại từ Zenodo và chạy các bước chuẩn bị — khoảng 30 phút.

In [ ]:
if os.path.exists("data/raw/A") and os.path.exists("data/processed/by_user/A.npz"):
    print("dữ liệu đã có, bỏ qua")
else:
    print("thiếu dữ liệu, chạy lại các bước chuẩn bị...")
    run("apt-get install -qq -y aria2")
    run("aria2c -x16 -s16 -k5M --summary-interval=0 --console-log-level=error "
        "-d /content -o tripod.zip "
        "https://zenodo.org/api/records/15022885/files/tripod.zip/content")
    run("mkdir -p data/raw && unzip -q -o /content/tripod.zip -d data/raw/")
    run("python scripts/1_organize_raw.py | tail -2")
    run("python scripts/2_make_npz.py | tail -2")
    run("python scripts/3_run_mobivital_prep.py 2>&1 | tail -2")

run("du -sh data/raw data/processed/*")

## 1. Dọn chỗ để chạy đúng lệnh của MobiVital

Code họ dùng đường dẫn tương đối `./dataset/mobivital/tripod/` và `./data_final/`,
nên hai thư mục đó phải nằm ngay trong repo họ. Script dựng bằng lối tắt rồi ghi
vào `.git/info/exclude` — loại trừ cục bộ, không thuộc repo — nên `git status`
của họ vẫn trống.

Việc khó nhất script này làm: **vá 52 tên file lỗi thời**. Bảng kết quả họ commit
sẵn ra đời trước khi dataset đổi tên lên Zenodo, 52 dòng ghi mốc tháng 10 còn bản
hiện tại là tháng 12. Chi tiết trong docstring của script.

In [ ]:
!python scripts/7_setup_mobivital.py

## 2. TN0a — chấm bảng MobiVital commit sẵn

Lệnh của họ, README mục *Detailed Analysis*. Thêm cờ `-d` (cũng là cờ của họ) để
trỏ sang thư mục có 52 lối tắt tên cũ.

Bậc này **chưa đụng model** — bin và phép đã ghi sẵn trong bảng, bỏ model nào vào
cũng ra số đó. Giá trị của nó: ra khớp `0.819` trong bài báo nghĩa là CSV mình
tải về đúng bằng CSV họ dùng.

In [ ]:
%cd /content/UWB_RADAR/external/mobivital
!python -m inference.evaluate -m tripod_mobivital_pre_invert_0.9.txt \
        -d ./dataset/mobivital/tripod_old_names

## 3. TN0b — checkpoint của họ, tự sinh bảng

Lệnh của họ, README mục *Evaluating MobiVital*. Dùng checkpoint
`lstm_pred_tripod_0.9.pth` họ phát hành.

Với mỗi buổi ghi: dựng 240 ứng viên (120 kênh × 2 phép), lọc bằng
`invert_detector`, cắt 52 cửa sổ mỗi ứng viên, cho LSTM dự báo, chọn kênh có tổng
Pearson cao nhất. Bước chọn **không nhìn nhịp thở thật** — đó là điểm chính của
bài báo.

In [ ]:
!python -m inference.mobivital_gen

Đổi tên bảng vừa sinh. `mobivital_gen.py` dòng 121 luôn ghi ra đúng một tên nên
chạy TN0c sẽ đè mất — README của họ viết `-m YOUR_METHOD.txt` chính là ngụ ý việc
đổi tên này.

In [ ]:
!mv inference/methods/tripod_mobivital_pre_invert_0.9.txt inference/methods/TN0b.txt
!wc -l < inference/methods/TN0b.txt
!python -m inference.evaluate -m TN0b.txt --save_file scores_TN0b.csv

## 4. TN0c — train lại từ đầu

Lệnh của họ, README mục *Training*. Thêm `--model_name` (cờ của họ) để file `.pth`
mới không đè lên checkpoint gốc — chính README của họ cảnh báo:

> *Running the training script using the default parameters will override the
> checkpoint. So back the default checkpoint up if necessary.*

Cấu hình lấy từ `checkpoints/optimal_params.json` của họ: 20 epoch, Adam lr 1e-4,
batch 64, MSE. Khoảng **20 phút trên GPU**.

In [ ]:
!python -m training.autoreg_training --model_name lstm_retrained

In [ ]:
!python -m inference.mobivital_gen --model_name lstm_retrained
!mv inference/methods/tripod_mobivital_pre_invert_0.9.txt inference/methods/TN0c.txt
!python -m inference.evaluate -m TN0c.txt --save_file scores_TN0c.csv

Ba lần chấm dùng ba `--save_file` riêng. Lý do: `evaluate.py` dòng 67 gán Series
vào DataFrame đã có, pandas căn theo index cũ và **vứt key lạ** — dồn chung một
file thì những lần sau bị cắt mất dòng.

## 5. Gom kết quả và kiểm tra không đụng code họ

In [ ]:
%cd /content/UWB_RADAR
!cp external/mobivital/inference/methods/TN0b.txt        results/
!cp external/mobivital/inference/methods/TN0c.txt        results/
!cp external/mobivital/inference/methods/scores_TN0b.csv results/
!cp external/mobivital/inference/methods/scores_TN0c.csv results/
!cp external/mobivital/inference/methods/scores.csv      results/scores_TN0a.csv
!ls -la results/
print()
print("KIỂM TRA không sửa gì trong repo MobiVital:")
run("git -C external/mobivital status --short || echo '  git status trống'")

## 6. TN0.1 — bộ chọn kênh của mình

Cùng checkpoint LSTM của MobiVital, nhưng chạy qua `src/scoring.py`.

In [ ]:
!python scripts/8_tn0_ours.py

## 7. Đối chiếu — cửa ải

Cùng checkpoint, cùng dữ liệu, cùng thuật toán, không có gì ngẫu nhiên → phải ra
y hệt. Script dừng hẳn nếu lệch.

Trùng 537/537 chứng minh cùng lúc ba điều:

| | vì sao suy ra được |
|---|---|
| dữ liệu `by_user/*.npz` đúng | `scoring.py` đọc `.npz` còn `mobivital_gen.py` đọc CSV |
| bộ chọn kênh đúng | 537/537 |
| hàm chấm điểm đúng | 537 điểm khớp tới chữ số 15 |

In [ ]:
!python scripts/9_tn0_compare.py

## 8. Cất kết quả lên Drive

Giữ nguyên cấu trúc thư mục để bung ở máy vào đúng chỗ:

```bash
tar -xzf ~/Downloads/tn0.tar.gz -C /Users/udnb/Desktop/THESIS_GRADUATE/
```

In [ ]:
run("mkdir -p /content/drive/MyDrive/mobivital")
run("tar -czf /content/drive/MyDrive/mobivital/tn0.tar.gz results")
run("ls -la /content/drive/MyDrive/mobivital/tn0.tar.gz")

## Xong

`results/` giờ đối xứng với pipeline gốc:

```
                 lua chon kenh      diem tung buoi ghi
MobiVital        TN0a.txt           scores_TN0a.csv
                 TN0b.txt           scores_TN0b.csv
                 TN0c.txt           scores_TN0c.csv
minh             TN0_1.txt          scores_TN0_1.csv
```

Từ đây thay LSTM bằng TCN, mọi khâu còn lại giữ nguyên.